# 01 · Data quality — PIHPS daily food prices

Before answering any business question, this notebook checks that the data can carry the
analysis: how complete it is, which values are wrong, and which days must be left out.

**Source:** PIHPS Nasional, Bank Indonesia — daily prices of 21 staple commodities,
traditional markets in 34 provinces plus national series for four market types,
2018-01-01 → 2026-09-30. Built with `uv run pangan build`.

In [1]:
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

from pangan import plots
from pangan.config import SQL_DIR, WAREHOUSE_PATH
from pangan.quality import quality_report

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
con = duckdb.connect(str(WAREHOUSE_PATH), read_only=True)


def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()


## 1. Volume and key integrity

In [2]:
report = quality_report(con)
coverage = report.pop("coverage")
pd.Series(report, name="value").to_frame()

,value
rows,1713633
duplicate_keys,0
null_prices,0
outliers,33


In [3]:
q('''
SELECT p.is_national, m.chain_order, m.name AS market_type, COUNT(*) AS rows,
       COUNT(DISTINCT f.commodity_id) AS commodities, MIN(f.date_key) AS first_day,
       MAX(f.date_key) AS last_day
FROM fact_price_daily f
JOIN dim_market_type m USING (market_type_id)
JOIN dim_province p USING (province_id)
GROUP BY ALL ORDER BY p.is_national, m.chain_order
''')

,is_national,chain_order,market_type,rows,commodities,first_day,last_day
0,False,3,Pasar Tradisional,1524446,21,2018-01-01,2026-09-30
1,True,1,Produsen,45953,21,2018-03-26,2026-09-30
2,True,2,Pedagang Besar,47838,21,2018-01-01,2026-09-30
3,True,3,Pasar Tradisional,47569,21,2018-01-01,2026-09-30
4,True,4,Pasar Modern,47827,21,2018-01-01,2026-09-30


## 2. Coverage: on how many business days does each province report?

PIHPS reports on working days. A province-year below ~80% would make monthly averages
unreliable.

In [4]:
plots.save_fig(fig := plots.plot_coverage(coverage), "coverage")
display(fig)
coverage["pct_business_days"].describe().round(3)

<Figure size 800x1072 with 2 Axes>

count    306.000
mean       0.966
std        0.027
min        0.920
25%        0.946
50%        0.954
75%        0.996
max        1.000
Name: pct_business_days, dtype: float64

## 3. Outliers: prices that are almost certainly typos

Rule: a price more than **3×** above or below the median of the same province, commodity and
market type within **±60 days** is flagged (`is_outlier`) and excluded from every view.

In [5]:
outliers = q('''
WITH ref AS (
    SELECT *, median(price_idr) OVER (
        PARTITION BY province_id, commodity_id, market_type_id ORDER BY date_key
        RANGE BETWEEN INTERVAL 60 DAYS PRECEDING AND INTERVAL 60 DAYS FOLLOWING
    ) AS reference_price
    FROM fact_price_daily
)
SELECT r.date_key, p.province_name, c.commodity_name, m.name AS market_type,
       r.price_idr, r.reference_price, ROUND(r.price_idr / r.reference_price, 2) AS ratio
FROM ref r
JOIN dim_province p USING (province_id)
JOIN dim_commodity c USING (commodity_id)
JOIN dim_market_type m USING (market_type_id)
WHERE r.is_outlier
ORDER BY r.date_key
''')
print(f"{len(outliers)} flagged prices")
outliers.groupby(["province_name", "commodity_name", "market_type"]).agg(
    days=("date_key", "count"), first=("date_key", "min"), last=("date_key", "max"),
    typical_ratio=("ratio", "median"),
).sort_values("days", ascending=False)

33 flagged prices


days      first       last  typical_ratio
province_name        commodity_name             market_type                                                 
Indonesia (Nasional) Daging Sapi Kualitas 2     Produsen             22 2021-09-20 2021-10-19           5.39
Papua Barat          Minyak Goreng Curah        Pasar Tradisional     4 2019-11-07 2020-03-02           5.60
                     Gula Pasir Lokal           Pasar Tradisional     2 2020-03-11 2020-03-24           3.30
DI Yogyakarta        Cabai Merah Keriting       Pasar Tradisional     1 2022-04-04 2022-04-04           3.89
Papua Barat          Bawang Putih Ukuran Sedang Pasar Tradisional     1 2020-06-26 2020-06-26           5.08
Indonesia (Nasional) Gula Pasir Lokal           Pasar Modern          1 2024-05-29 2024-05-29           3.17
Papua Barat          Cabai Rawit Hijau          Pasar Tradisional     1 2021-01-04 2021-01-04           3.37
                     Beras Kualitas Super II    Pasar Tradisional     1 2020-07-14 2020-07-14           3.07

Why a **rolling** window and not the calendar month? Plot the series with the longest typo run:

In [6]:
beef = q('''
SELECT date_key, price_idr, is_outlier FROM fact_price_daily
WHERE province_id = 0 AND market_type_id = 4 AND commodity_id = 'com_9'
  AND date_key BETWEEN '2021-08-01' AND '2021-11-30'
ORDER BY date_key
''')
beef.assign(month=beef.date_key.dt.to_period("M")).groupby(["month", "is_outlier"]).price_idr.agg(
    ["count", "median"]
)

count    median
month   is_outlier                 
2021-08 False          22  107450.0
2021-09 False          13  107350.0
        True            9  583050.0
2021-10 False           8  108150.0
        True           13  583050.0
2021-11 False          22  108150.0

## 4. Thin days: when the national figure is not national

In [7]:
thin = q('''
SELECT year, COUNT(*) FILTER (WHERE is_thin_day AND dow <= 5) AS thin_weekdays
FROM dim_date GROUP BY year ORDER BY year
''')
thin.set_index("year").T

year,2018,2019,2020,2021,2022,2023,2024,2025,2026
thin_weekdays,16,12,20,13,14,2,0,0,0


In [8]:
q('''
SELECT d.date_key, d.reporting_provinces, d.is_thin_day, f.price_idr AS national_beras_bawah_I
FROM dim_date d
LEFT JOIN fact_price_daily f
  ON f.date_key = d.date_key AND f.province_id = 0 AND f.market_type_id = 1
 AND f.commodity_id = 'com_1'
WHERE d.date_key BETWEEN '2022-04-26' AND '2022-05-11' AND d.dow <= 5
ORDER BY d.date_key
''')

,date_key,reporting_provinces,is_thin_day,national_beras_bawah_I
0,2022-04-26,34,False,10700
1,2022-04-27,34,False,10700
2,2022-04-28,34,False,10700
3,2022-04-29,2,True,12750
4,2022-05-02,0,True,<NA>
5,2022-05-03,0,True,<NA>
6,2022-05-04,1,True,13200
7,2022-05-05,2,True,11600
8,2022-05-06,2,True,11600
9,2022-05-09,34,False,10700
